# DeepTruth — Phase 1: Data Preparation

**Run this entire notebook in Google Colab Pro.**

What this notebook does:
1. Mounts Google Drive and creates the folder structure
2. Installs all required packages
3. Downloads GenImage subset (DALL-E, Stable Diffusion, Midjourney)
4. Downloads CelebDF-v2 videos
5. Downloads FaceForensics++ compressed videos (requires your download.py from email)
6. Downloads DFDC Preview (requires Kaggle API key)
7. Extracts frames from all video datasets
8. Crops and aligns faces using MTCNN on every image
9. Saves the final clean dataset to Google Drive

**Before running:** Complete these 3 steps on your local machine:
- Free up 150GB on Google Drive
- Have your `kaggle.json` API key ready
- Have your FaceForensics++ `download.py` script ready (from the email)

**Estimated total time:** 4-6 hours (mostly downloading)

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Root folder for everything — change this if you want a different location
ROOT = '/content/drive/MyDrive/DeepTruth'

# Create full folder structure
folders = [
    f'{ROOT}/data/raw/real',
    f'{ROOT}/data/raw/fake/stylegan2',
    f'{ROOT}/data/raw/fake/genimage/dalle',
    f'{ROOT}/data/raw/fake/genimage/stable_diffusion',
    f'{ROOT}/data/raw/fake/genimage/midjourney',
    f'{ROOT}/data/raw/fake/celebdf',
    f'{ROOT}/data/raw/fake/faceforensics',
    f'{ROOT}/data/raw/fake/dfdc',
    f'{ROOT}/data/raw/videos/celebdf',
    f'{ROOT}/data/raw/videos/faceforensics',
    f'{ROOT}/data/raw/videos/dfdc',
    f'{ROOT}/data/raw/videos/dfd_real',
    f'{ROOT}/data/raw/videos/dfd_fake',
    f'{ROOT}/data/processed/train/real',
    f'{ROOT}/data/processed/train/fake',
    f'{ROOT}/data/processed/val/real',
    f'{ROOT}/data/processed/val/fake',
    f'{ROOT}/data/processed/test/real',
    f'{ROOT}/data/processed/test/fake',
    f'{ROOT}/data/processed/video_sequences/real',
    f'{ROOT}/data/processed/video_sequences/fake',
    f'{ROOT}/models',
    f'{ROOT}/checkpoints',
    f'{ROOT}/logs',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print('Drive mounted and folder structure created.')
print(f'Root: {ROOT}')

# Check available space
import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free space: {free / (1024**3):.1f} GB')
if free / (1024**3) < 100:
    print('WARNING: Less than 100GB free. Consider clearing space before continuing.')
else:
    print('Space is sufficient.')

## Cell 2 — Install Packages

In [ ]:
!pip install -q \
    facenet-pytorch \
    opencv-python-headless \
    gdown \
    huggingface_hub \
    kaggle \
    tqdm \
    Pillow \
    numpy \
    albumentations

print('All packages installed.')

## Cell 3 — Upload Existing Data from Local Machine (FFHQ + StyleGAN2 + DFD Videos)

You already have 50K real images, 50K fake images, and 3,431 DFD videos on your local machine at `C:/Users/hp/Desktop/DeepTruth/data/raw/`.

**Two options to get them into Colab:**

**Option A (Recommended) — Upload via Google Drive desktop app:**
1. Install Google Drive desktop app on your PC if not already installed
2. Copy `data/raw/real/` → `DeepTruth/data/raw/real/` in your Drive folder
3. Copy `data/raw/fake/` → `DeepTruth/data/raw/fake/stylegan2/` in your Drive folder
4. Copy `data/raw/video/DFD_original_sequences/` → `DeepTruth/data/raw/videos/dfd_real/`
5. Copy `data/raw/video/DFD_manipulated_sequences/` → `DeepTruth/data/raw/videos/dfd_fake/`
6. Wait for sync to complete, then continue

**Option B — Zip and upload manually:**

In [ ]:
# Run this cell to verify your existing data is visible in Drive
import os

def count_files(path, extensions=('.jpg', '.jpeg', '.png', '.mp4', '.avi', '.mov')):
    if not os.path.exists(path):
        return 0
    count = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.lower().endswith(extensions):
                count += 1
    return count

print('Existing data check:')
print(f'  Real images:        {count_files(ROOT + "/data/raw/real"):,}')
print(f'  StyleGAN2 fakes:    {count_files(ROOT + "/data/raw/fake/stylegan2"):,}')
print(f'  DFD real videos:    {count_files(ROOT + "/data/raw/videos/dfd_real", (".mp4", ".avi", ".mov")):,}')
print(f'  DFD fake videos:    {count_files(ROOT + "/data/raw/videos/dfd_fake", (".mp4", ".avi", ".mov")):,}')
print()
print('Target: 50K real, 50K fake, ~250 real videos, ~3000 fake videos')
print('If counts are 0, go back and sync your local data to Drive first.')

## Cell 4 — Setup Kaggle API Key

Paste the contents of your `kaggle.json` file below.

In [ ]:
import json
import os

# PASTE your kaggle.json contents here:
kaggle_credentials = {
    "username": "YOUR_KAGGLE_USERNAME",
    "key": "YOUR_KAGGLE_API_KEY"
}

# Save to the expected location
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Verify
!kaggle datasets list --search "deepfake" -p 1
print('Kaggle API configured successfully.')

## Cell 5 — Download GenImage Subset (DALL-E, Stable Diffusion, Midjourney)

This fills the biggest gap — your model has never seen diffusion-generated images.
Downloads ~8-15GB. Takes 30-60 minutes.

In [ ]:
from huggingface_hub import snapshot_download
import os

GENIMAGE_DIR = f'{ROOT}/data/raw/fake/genimage'

# Download the three most important generators
# stable_diffusion_v_1_4: most widely used, strong baseline
# dalle3: covers ChatGPT image generation
# midjourney: covers highest-quality AI images

generators_to_download = [
    'stable_diffusion_v_1_4',
    'dalle3',
    'midjourney',
]

print('Downloading GenImage subset from HuggingFace...')
print('This will take 30-60 minutes depending on connection speed.')

for generator in generators_to_download:
    print(f'\nDownloading {generator}...')
    try:
        snapshot_download(
            repo_id='GenImage/GenImage',
            repo_type='dataset',
            allow_patterns=[f'{generator}/*'],
            local_dir=GENIMAGE_DIR,
            local_dir_use_symlinks=False,
        )
        print(f'{generator} downloaded.')
    except Exception as e:
        print(f'Error downloading {generator}: {e}')
        print('Trying alternative download method...')
        # Fallback: download via kaggle
        !kaggle datasets download -d datasets/genimage-{generator} -p {GENIMAGE_DIR} --unzip

# Count what we got
total = 0
for generator in generators_to_download:
    path = os.path.join(GENIMAGE_DIR, generator)
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f'{generator}: {count:,} images')
        total += count

print(f'\nTotal GenImage fake images downloaded: {total:,}')

## Cell 6 — Download CelebDF-v2

High-quality celebrity face swap videos. ~2.2GB. Takes 5-10 minutes.

In [ ]:
import gdown
import os
import zipfile

CELEBDF_DIR = f'{ROOT}/data/raw/videos/celebdf'
os.makedirs(CELEBDF_DIR, exist_ok=True)

print('Downloading CelebDF-v2...')

# CelebDF-v2 Google Drive folder
# Real videos
print('Downloading real videos...')
gdown.download_folder(
    'https://drive.google.com/drive/folders/1iml8GVEK1LHLiD8wkASVBIYbFUNvHjSM',
    output=f'{CELEBDF_DIR}/real',
    quiet=False
)

# Fake videos (Celeb-synthesis)
print('Downloading fake videos...')
gdown.download_folder(
    'https://drive.google.com/drive/folders/1KFHOKKBEUvPGBrB0jM82URP7j1ZMsFWe',
    output=f'{CELEBDF_DIR}/fake',
    quiet=False
)

real_count = len([f for f in os.listdir(f'{CELEBDF_DIR}/real') if f.endswith('.mp4')])
fake_count = len([f for f in os.listdir(f'{CELEBDF_DIR}/fake') if f.endswith('.mp4')])
print(f'CelebDF-v2: {real_count} real videos, {fake_count} fake videos')

## Cell 7 — Download FaceForensics++ (compressed, c23)

**You need the `download.py` script from the FaceForensics++ email.**

Upload your `download.py` to `/content/` using the Colab file browser (left panel → Files → Upload), then run this cell.

In [ ]:
import os

FF_DIR = f'{ROOT}/data/raw/videos/faceforensics'
os.makedirs(FF_DIR, exist_ok=True)

# Check that download.py was uploaded
if not os.path.exists('/content/download.py'):
    print('ERROR: download.py not found.')
    print('Upload it via the Colab file browser (left panel → Files → Upload icon)')
else:
    print('download.py found. Starting download...')
    print('Downloading FaceForensics++ compressed (c23) — ~10GB, ~30-60 min...')

    # Download all manipulation types, compressed version (c23)
    # -d: dataset split (all)
    # -c c23: compression level (most practical size)
    # -t videos: download videos not images
    !python /content/download.py \
        {FF_DIR} \
        -d all \
        -c c23 \
        -t videos \
        --server EU

    # Count what we got
    for subdir in ['original_sequences', 'manipulated_sequences']:
        path = os.path.join(FF_DIR, subdir)
        if os.path.exists(path):
            count = sum(len(files) for _, _, files in os.walk(path))
            print(f'{subdir}: {count} files')

## Cell 8 — Download DFDC Preview (DeepFake Detection Challenge)

Diverse real-world deepfakes from Meta's challenge. ~10GB. Requires Kaggle API.

In [ ]:
import os

DFDC_DIR = f'{ROOT}/data/raw/videos/dfdc'
os.makedirs(DFDC_DIR, exist_ok=True)

print('Downloading DFDC Preview dataset from Kaggle...')
print('~10GB — takes 20-40 minutes...')

!kaggle datasets download \
    -d bardofcodes/deepfake-detection-challenge-dfdcp \
    -p {DFDC_DIR} \
    --unzip

# Count videos
video_count = 0
for root, dirs, files in os.walk(DFDC_DIR):
    for f in files:
        if f.endswith(('.mp4', '.avi')):
            video_count += 1

print(f'DFDC: {video_count} videos downloaded')

## Cell 9 — Frame Extraction from All Video Datasets

Extracts evenly-spaced frames from every video in all datasets.
Each video yields multiple labeled face images for training.

Settings:
- 32 frames per video (for temporal model training)
- Saves both the full frames AND sequences for the temporal stream

In [ ]:
import cv2
import os
import numpy as np
from tqdm import tqdm

FRAMES_PER_VIDEO = 32

def extract_frames(video_path, output_dir, n_frames=FRAMES_PER_VIDEO):
    """Extract n evenly-spaced frames from a video."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return 0

    indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    saved = 0

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    os.makedirs(output_dir, exist_ok=True)

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            out_path = os.path.join(output_dir, f'{video_name}_frame_{idx:06d}.jpg')
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1

    cap.release()
    return saved


def process_video_dataset(video_dir, output_dir, label, extensions=('.mp4', '.avi', '.mov')):
    """Process all videos in a directory tree."""
    video_files = []
    for root, dirs, files in os.walk(video_dir):
        for f in files:
            if f.lower().endswith(extensions):
                video_files.append(os.path.join(root, f))

    if not video_files:
        print(f'  No videos found in {video_dir}')
        return 0

    os.makedirs(output_dir, exist_ok=True)
    total_frames = 0

    for video_path in tqdm(video_files, desc=f'Extracting {label}'):
        n = extract_frames(video_path, output_dir)
        total_frames += n

    print(f'  {label}: {len(video_files)} videos → {total_frames:,} frames')
    return total_frames


# Define all video datasets to process
# Format: (input_video_dir, output_frames_dir, label)
TEMP_FRAMES = '/content/frames_temp'

video_datasets = [
    # DFD (already have)
    (f'{ROOT}/data/raw/videos/dfd_real',       f'{TEMP_FRAMES}/dfd/real',          'DFD Real'),
    (f'{ROOT}/data/raw/videos/dfd_fake',       f'{TEMP_FRAMES}/dfd/fake',          'DFD Fake'),
    # CelebDF-v2
    (f'{ROOT}/data/raw/videos/celebdf/real',   f'{TEMP_FRAMES}/celebdf/real',      'CelebDF Real'),
    (f'{ROOT}/data/raw/videos/celebdf/fake',   f'{TEMP_FRAMES}/celebdf/fake',      'CelebDF Fake'),
    # FaceForensics++
    (f'{ROOT}/data/raw/videos/faceforensics/original_sequences',    f'{TEMP_FRAMES}/ff/real', 'FF++ Real'),
    (f'{ROOT}/data/raw/videos/faceforensics/manipulated_sequences', f'{TEMP_FRAMES}/ff/fake', 'FF++ Fake'),
    # DFDC
    (f'{ROOT}/data/raw/videos/dfdc',           f'{TEMP_FRAMES}/dfdc/mixed',        'DFDC'),
]

print('Starting frame extraction from all video datasets...')
print('This will take 1-2 hours depending on dataset size.\n')

total_real = 0
total_fake = 0

for video_dir, output_dir, label in video_datasets:
    if os.path.exists(video_dir):
        n = process_video_dataset(video_dir, output_dir, label)
        if 'real' in label.lower() or 'Real' in label:
            total_real += n
        else:
            total_fake += n
    else:
        print(f'  Skipping {label} — directory not found: {video_dir}')

print(f'\nFrame extraction complete.')
print(f'Total real frames: {total_real:,}')
print(f'Total fake frames: {total_fake:,}')

## Cell 10 — MTCNN Face Detection and Cropping

Detects and crops the face region from every image (both static images and extracted video frames).
Aligns faces to a standard orientation. Resizes to 224×224.

Images where no face is detected are discarded — they can't be used for face deepfake detection.

In [ ]:
import torch
from facenet_pytorch import MTCNN
from PIL import Image
import os
import numpy as np
from tqdm import tqdm

# Use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Initialize MTCNN
# margin=20: adds padding around face crop
# image_size=224: output size matches our model input
# keep_all=False: only take the largest/most confident face per image
mtcnn = MTCNN(
    image_size=224,
    margin=20,
    keep_all=False,
    min_face_size=40,
    device=device
)


def crop_faces_in_dir(input_dir, output_dir, batch_size=32):
    """
    Run MTCNN face detection on all images in input_dir.
    Saves cropped, aligned faces to output_dir.
    Returns (saved_count, skipped_count).
    """
    if not os.path.exists(input_dir):
        return 0, 0

    os.makedirs(output_dir, exist_ok=True)

    image_files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    saved = 0
    skipped = 0

    for i in tqdm(range(0, len(image_files), batch_size), desc=f'MTCNN {os.path.basename(input_dir)}'):
        batch_files = image_files[i:i+batch_size]
        batch_images = []
        valid_files = []

        for fname in batch_files:
            try:
                img = Image.open(os.path.join(input_dir, fname)).convert('RGB')
                batch_images.append(img)
                valid_files.append(fname)
            except Exception:
                skipped += 1
                continue

        if not batch_images:
            continue

        try:
            faces = mtcnn(batch_images)
        except Exception:
            skipped += len(batch_images)
            continue

        for fname, face in zip(valid_files, faces):
            if face is None:
                skipped += 1
                continue

            # Convert tensor (C, H, W) back to PIL image
            # MTCNN returns values in [-1, 1], convert to [0, 255]
            face_np = face.permute(1, 2, 0).numpy()
            face_np = ((face_np + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            face_img = Image.fromarray(face_np)

            out_path = os.path.join(output_dir, fname)
            face_img.save(out_path, quality=95)
            saved += 1

    return saved, skipped


print('Running MTCNN face detection on all datasets...')
print('This is the longest step — 1-3 hours on GPU.\n')

# All input→output directory pairs to process
PROCESSED = f'{ROOT}/data/processed_faces'

face_crop_tasks = [
    # Static image datasets (already on Drive)
    (f'{ROOT}/data/raw/real',                f'{PROCESSED}/real/ffhq',           'FFHQ Real'),
    (f'{ROOT}/data/raw/fake/stylegan2',      f'{PROCESSED}/fake/stylegan2',      'StyleGAN2 Fake'),
    # GenImage (diffusion)
    (f'{ROOT}/data/raw/fake/genimage/dalle',            f'{PROCESSED}/fake/dalle',            'DALL-E'),
    (f'{ROOT}/data/raw/fake/genimage/stable_diffusion', f'{PROCESSED}/fake/stable_diffusion', 'Stable Diffusion'),
    (f'{ROOT}/data/raw/fake/genimage/midjourney',       f'{PROCESSED}/fake/midjourney',       'Midjourney'),
    # Video frames
    (f'{TEMP_FRAMES}/dfd/real',     f'{PROCESSED}/real/dfd',     'DFD Real Frames'),
    (f'{TEMP_FRAMES}/dfd/fake',     f'{PROCESSED}/fake/dfd',     'DFD Fake Frames'),
    (f'{TEMP_FRAMES}/celebdf/real', f'{PROCESSED}/real/celebdf', 'CelebDF Real Frames'),
    (f'{TEMP_FRAMES}/celebdf/fake', f'{PROCESSED}/fake/celebdf', 'CelebDF Fake Frames'),
    (f'{TEMP_FRAMES}/ff/real',      f'{PROCESSED}/real/ff',      'FF++ Real Frames'),
    (f'{TEMP_FRAMES}/ff/fake',      f'{PROCESSED}/fake/ff',      'FF++ Fake Frames'),
    (f'{TEMP_FRAMES}/dfdc/mixed',   f'{PROCESSED}/fake/dfdc',    'DFDC Frames'),
]

summary = []
for input_dir, output_dir, label in face_crop_tasks:
    s, sk = crop_faces_in_dir(input_dir, output_dir)
    summary.append((label, s, sk))
    print(f'{label}: {s:,} faces saved, {sk:,} skipped (no face detected)')

print('\n=== MTCNN Complete ===')
total_saved = sum(s for _, s, _ in summary)
total_skipped = sum(sk for _, _, sk in summary)
print(f'Total face images: {total_saved:,}')
print(f'Total discarded:   {total_skipped:,}')

## Cell 11 — Build Train / Val / Test Splits

Combines all processed face images into a unified split.
Ratio: 80% train, 10% val, 10% test.
Balances real and fake counts per split.

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

random.seed(42)

PROCESSED = f'{ROOT}/data/processed_faces'
SPLIT_DIR = f'{ROOT}/data/splits'

for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        os.makedirs(f'{SPLIT_DIR}/{split}/{label}', exist_ok=True)


def collect_all_images(base_dir):
    images = []
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                images.append(os.path.join(root, f))
    return images


def split_and_copy(images, output_base, label, train_ratio=0.8, val_ratio=0.1):
    random.shuffle(images)
    n = len(images)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    splits = {
        'train': images[:train_end],
        'val':   images[train_end:val_end],
        'test':  images[val_end:],
    }

    for split_name, split_files in splits.items():
        out_dir = f'{output_base}/{split_name}/{label}'
        for src in tqdm(split_files, desc=f'{label} → {split_name}', leave=False):
            fname = os.path.basename(src)
            # Prefix with source subdir to avoid filename collisions
            source_tag = os.path.basename(os.path.dirname(src))
            dst = os.path.join(out_dir, f'{source_tag}_{fname}')
            shutil.copy2(src, dst)

    return {k: len(v) for k, v in splits.items()}


print('Collecting all real images...')
all_real = collect_all_images(f'{PROCESSED}/real')
print(f'Total real face images: {len(all_real):,}')

print('Collecting all fake images...')
all_fake = collect_all_images(f'{PROCESSED}/fake')
print(f'Total fake face images: {len(all_fake):,}')

# Balance: cap both at the size of the smaller class
min_count = min(len(all_real), len(all_fake))
print(f'\nBalancing dataset to {min_count:,} per class...')
all_real = random.sample(all_real, min_count)
all_fake = random.sample(all_fake, min_count)

print('Creating train/val/test splits...')
real_counts = split_and_copy(all_real, SPLIT_DIR, 'real')
fake_counts = split_and_copy(all_fake, SPLIT_DIR, 'fake')

print('\n=== Final Dataset Split ===')
for split in ['train', 'val', 'test']:
    print(f'{split:5s}: {real_counts[split]:,} real + {fake_counts[split]:,} fake = {real_counts[split] + fake_counts[split]:,} total')

## Cell 12 — Build Video Sequences for Temporal Training

For the temporal stream, we need sequences of 32 consecutive face frames from the same video.
This cell saves `.npz` files, each containing a (32, 224, 224, 3) array from one video.

In [ ]:
import cv2
import numpy as np
import os
from tqdm import tqdm
import torch
from facenet_pytorch import MTCNN
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn = MTCNN(image_size=224, margin=20, keep_all=False, device=device)

SEQ_DIR = f'{ROOT}/data/sequences'
os.makedirs(f'{SEQ_DIR}/real', exist_ok=True)
os.makedirs(f'{SEQ_DIR}/fake', exist_ok=True)

N_FRAMES = 32


def extract_face_sequence(video_path, n_frames=N_FRAMES):
    """
    Extract n_frames evenly-spaced face crops from a video.
    Returns numpy array of shape (n_frames, 224, 224, 3) or None.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < n_frames:
        cap.release()
        return None

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            frames.append(None)
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame_rgb))

    cap.release()

    if any(f is None for f in frames):
        return None

    # Run MTCNN on all frames
    try:
        faces = mtcnn(frames)
    except Exception:
        return None

    if any(f is None for f in faces):
        return None

    # Convert to numpy (n_frames, 224, 224, 3), values in [0, 255]
    sequence = []
    for face in faces:
        face_np = face.permute(1, 2, 0).numpy()
        face_np = ((face_np + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
        sequence.append(face_np)

    return np.stack(sequence, axis=0)


def process_video_dir_to_sequences(video_dir, output_dir, label, max_videos=500):
    videos = []
    for root, dirs, files in os.walk(video_dir):
        for f in files:
            if f.lower().endswith(('.mp4', '.avi', '.mov')):
                videos.append(os.path.join(root, f))

    videos = videos[:max_videos]
    saved = 0

    for video_path in tqdm(videos, desc=f'Sequences {label}'):
        seq = extract_face_sequence(video_path)
        if seq is None:
            continue
        name = os.path.splitext(os.path.basename(video_path))[0]
        np.savez_compressed(
            os.path.join(output_dir, f'{name}.npz'),
            frames=seq
        )
        saved += 1

    print(f'{label}: {saved}/{len(videos)} videos → sequences saved')
    return saved


video_sequence_tasks = [
    (f'{ROOT}/data/raw/videos/dfd_real',       f'{SEQ_DIR}/real', 'DFD Real',     500),
    (f'{ROOT}/data/raw/videos/dfd_fake',       f'{SEQ_DIR}/fake', 'DFD Fake',     500),
    (f'{ROOT}/data/raw/videos/celebdf/real',   f'{SEQ_DIR}/real', 'CelebDF Real', 300),
    (f'{ROOT}/data/raw/videos/celebdf/fake',   f'{SEQ_DIR}/fake', 'CelebDF Fake', 300),
    (f'{ROOT}/data/raw/videos/faceforensics/original_sequences',    f'{SEQ_DIR}/real', 'FF++ Real', 400),
    (f'{ROOT}/data/raw/videos/faceforensics/manipulated_sequences', f'{SEQ_DIR}/fake', 'FF++ Fake', 400),
    (f'{ROOT}/data/raw/videos/dfdc',           f'{SEQ_DIR}/fake', 'DFDC',         400),
]

print('Building video sequences for temporal training...')
print('Each sequence = 32 aligned face frames from one video = one training sample\n')

for video_dir, output_dir, label, max_v in video_sequence_tasks:
    if os.path.exists(video_dir):
        process_video_dir_to_sequences(video_dir, output_dir, label, max_v)
    else:
        print(f'Skipping {label} — not found')

real_seqs = len(os.listdir(f'{SEQ_DIR}/real'))
fake_seqs = len(os.listdir(f'{SEQ_DIR}/fake'))
print(f'\nTotal sequences: {real_seqs} real, {fake_seqs} fake')

## Cell 13 — Dataset Summary and Sanity Check

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

SPLIT_DIR = f'{ROOT}/data/splits'
SEQ_DIR   = f'{ROOT}/data/sequences'

print('=== FINAL DATASET SUMMARY ===')
print()

total_images = 0
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        path = f'{SPLIT_DIR}/{split}/{label}'
        if os.path.exists(path):
            n = len(os.listdir(path))
            print(f'  {split:5s} / {label:4s}: {n:,} images')
            total_images += n

print(f'\n  Total image samples: {total_images:,}')

real_seqs = len(os.listdir(f'{SEQ_DIR}/real')) if os.path.exists(f'{SEQ_DIR}/real') else 0
fake_seqs = len(os.listdir(f'{SEQ_DIR}/fake')) if os.path.exists(f'{SEQ_DIR}/fake') else 0
print(f'  Total video sequences: {real_seqs + fake_seqs:,} ({real_seqs} real, {fake_seqs} fake)')

print()
print('=== SANITY CHECK — Sample Images ===')

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Face Crops (Top: Real, Bottom: Fake)', fontsize=14)

for i, label in enumerate(['real', 'fake']):
    path = f'{SPLIT_DIR}/train/{label}'
    if not os.path.exists(path):
        continue
    samples = random.sample(os.listdir(path), min(5, len(os.listdir(path))))
    for j, fname in enumerate(samples):
        img = Image.open(os.path.join(path, fname)).convert('RGB')
        axes[i][j].imshow(img)
        axes[i][j].axis('off')
        axes[i][j].set_title(label.upper(), fontsize=9)

plt.tight_layout()
plt.savefig(f'{ROOT}/data/sample_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print('Sample grid saved to Drive.')

print()
print('=== SANITY CHECK — Video Sequence ===')
seq_files = os.listdir(f'{SEQ_DIR}/fake')
if seq_files:
    sample_seq = np.load(os.path.join(f'{SEQ_DIR}/fake', seq_files[0]))['frames']
    print(f'Sequence shape: {sample_seq.shape}  (should be (32, 224, 224, 3))')
    print(f'Value range: [{sample_seq.min()}, {sample_seq.max()}]  (should be [0, 255])')

print()
print('Phase 1 complete. Move on to 02_build_model.ipynb')